In [39]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict,Literal,Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
import operator
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

True

In [40]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [41]:
model = ChatGroq(model = "llama-3.3-70b-versatile")

In [42]:
def chat_node(state:ChatState):
    messages = state['messages']
    response = model.invoke(messages)
    return {'messages':[response]}


In [43]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

workflow = graph.compile(checkpointer=checkpointer)

In [44]:
thread_id = "1"

while True:
    user_input = input("You: ")
    print(user_input)
    if user_input == "exit" or user_input == "quit" or user_input == "stop":
        break
    config = {"configurable":{"thread_id":thread_id}}
    initial_state = {"messages":[HumanMessage(content=user_input)]}
    output = workflow.invoke(initial_state,config=config)
    print("AI: ",output['messages'][-1].content)

Hi my name is Ravi
AI:  Hello Ravi! It's nice to meet you. Is there something I can help you with or would you like to chat?
Tell me what is capital of India
AI:  The capital of India is **New Delhi**.
Tell me What was our last Question
AI:  Our last question was about the capital of India, and I answered that it is **New Delhi**. Before that, you had introduced yourself and told me your name is **Ravi**.
good very good now stop
AI:  It was nice chatting with you, Ravi. Goodbye!
stop


In [45]:
workflow.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi my name is Ravi', additional_kwargs={}, response_metadata={}, id='45cedbca-243f-4bea-b079-249a24925171'), AIMessage(content="Hello Ravi! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 41, 'total_tokens': 68, 'completion_time': 0.050213677, 'completion_tokens_details': None, 'prompt_time': 0.003739204, 'prompt_tokens_details': None, 'queue_time': 0.058050035, 'total_time': 0.053952881}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f4f11-af30-7681-85fb-f79b934b8858-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 27, 'total_tokens': 68}), HumanMessage(content='Tell me what is capital of India', ad